# Notebook 03 — Extracción de Embeddings

**Prerequisito:** `artifacts/checkpoints/encoder_best.pt` debe existir (entrenado en notebook 02 o en Colab).

Este notebook genera los embeddings de 1024 dimensiones para train/val/test
y los guarda en `data/embeddings/`. Todos los modelos clásicos usarán exactamente estos embeddings.

## Cómo ejecutar
```bash
jupyter lab notebooks/03_extract_embeddings.ipynb
# O directamente:
python -m scripts.extract_embeddings --checkpoint artifacts/checkpoints/encoder_best.pt
```

In [ ]:
# ════════════════════════════════════════════════════════════════
# SETUP — Colab sin auto-push (clone público con destino absoluto)
# ════════════════════════════════════════════════════════════════
import os, sys, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    REPO     = "Malaria-Dectetion-Deeplearning"
    REPO_DIR = f"/content/{REPO}"
    REPO_URL = "https://github.com/JuanCOD001116/Malaria-Dectetion-Deeplearning.git"

    os.chdir("/content")

    for stray in (f"/{REPO}", REPO_DIR):
        if os.path.exists(stray) and not os.path.exists(f"{stray}/.git"):
            shutil.rmtree(stray, ignore_errors=True)

    if not os.path.exists(f"{REPO_DIR}/.git"):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")

    assert os.path.exists(f"{REPO_DIR}/.git"), f"git clone falló: {REPO_DIR}"

    get_ipython().run_line_magic("cd", REPO_DIR)
    get_ipython().system("git pull origin main")
    get_ipython().system("pip install -r requirements.txt -q")

    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive/malaria_project"
    get_ipython().system(f"mkdir -p {DRIVE_ROOT}/checkpoints {DRIVE_ROOT}/embeddings")
    get_ipython().system("rm -rf artifacts/checkpoints data/embeddings")
    get_ipython().system("mkdir -p artifacts data")
    get_ipython().system(f"ln -sfn {DRIVE_ROOT}/checkpoints artifacts/checkpoints")
    get_ipython().system(f"ln -sfn {DRIVE_ROOT}/embeddings   data/embeddings")
    get_ipython().system("mkdir -p artifacts/figures artifacts/metrics artifacts/logs data/processed")
    print("✓ Colab listo. Pesados → Drive, ligeros → repo local.")

cwd = Path().resolve()
REPO_ROOT = cwd if (cwd / "src").exists() else cwd.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(f"Working dir: {REPO_ROOT}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# DATASET — Kaggle API (necesario porque extract_embeddings abre imágenes)
# ════════════════════════════════════════════════════════════════
if IN_COLAB and not os.path.exists("cell_images"):
    if os.path.exists(f"{DRIVE_ROOT}/kaggle.json"):
        get_ipython().system("mkdir -p ~/.kaggle")
        get_ipython().system(f"cp {DRIVE_ROOT}/kaggle.json ~/.kaggle/")
    else:
        from google.colab import files
        print("Sube tu kaggle.json (Kaggle → Settings → Create API Token):")
        files.upload()
        get_ipython().system("mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/")
        get_ipython().system(f"cp ~/.kaggle/kaggle.json {DRIVE_ROOT}/kaggle.json")
    get_ipython().system("chmod 600 ~/.kaggle/kaggle.json")
    get_ipython().system("pip install kaggle -q")
    get_ipython().system("kaggle datasets download -d iarunava/cell-images-for-detecting-malaria -q")
    get_ipython().system("unzip -q cell-images-for-detecting-malaria.zip")
    get_ipython().system("rm -f cell-images-for-detecting-malaria.zip")
    get_ipython().system("if [ -d cell_images/cell_images ]; then mv cell_images/cell_images/* cell_images/ 2>/dev/null; rmdir cell_images/cell_images 2>/dev/null; fi")
    print(f"✓ Dataset listo: {len(os.listdir('cell_images/Parasitized'))} parasitized")

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from src.utils.seed import set_global_seed
from src.utils.io import load_config, load_checkpoint, save_embeddings
from src.data.augmentations import get_eval_transform
from src.data.dataset import MalariaDataset
from src.models.encoder import ContrastiveEncoder

set_global_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

In [ ]:
CHECKPOINT_PATH = 'artifacts/checkpoints/encoder_best.pt'
assert Path(CHECKPOINT_PATH).exists(), f'No existe {CHECKPOINT_PATH}. Entrena primero con notebook 02.'

cfg = load_config('configs/contrastive.yaml')
data_cfg = load_config('configs/data.yaml')

enc_cfg = cfg.get('encoder', {})
model = ContrastiveEncoder(
    embedding_dim=enc_cfg.get('embedding_dim', 1024),
    proj_dim=enc_cfg.get('proj_dim', 128),
    pretrained=False,
).to(device)

ckpt = load_checkpoint(CHECKPOINT_PATH, device=str(device))
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Checkpoint cargado — epoch {ckpt.get("epoch", "?")} | val_loss={ckpt.get("val_loss", "?"): .4f}')

In [ ]:
transform = get_eval_transform(img_size=data_cfg.get('img_size', 96))
processed_dir = Path(data_cfg['processed_dir'])
embeddings_dir = Path(data_cfg['embeddings_dir'])
embeddings_dir.mkdir(parents=True, exist_ok=True)

for split in ['train', 'val', 'test']:
    ds = MalariaDataset(processed_dir / f'{split}.csv', transform=transform)
    loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=0)
    
    all_emb, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=f'  {split}'):
            emb = model(imgs.to(device), return_embedding=True).cpu().numpy()
            all_emb.append(emb)
            all_labels.append(labels.numpy())
    
    X = np.concatenate(all_emb)
    y = np.concatenate(all_labels)
    assert X.shape[1] == 1024, f'Error: embedding dim={X.shape[1]}'
    save_embeddings(X, y, split, embeddings_dir)
    print(f'{split}: {X.shape} | pos={y.sum()}/{len(y)}')

print('\nEmbeddings guardados en', embeddings_dir.resolve())

In [ ]:
# ════════════════════════════════════════════════════════════════
# DESCARGA MANUAL — baja el .ipynb ejecutado a tu PC
# Sin auto-push: tú subes/entregas el notebook por el medio que prefieras.
# ════════════════════════════════════════════════════════════════
if IN_COLAB:
    NOTEBOOK = "03_extract_embeddings"
    # Forzar guardado del .ipynb (preserva outputs y figuras embebidas)
    try:
        from google.colab import _message
        _message.blocking_request("save_notebook", request="", timeout_sec=10)
    except Exception:
        pass
    # Descargar a tu PC (revisa carpeta de descargas)
    from google.colab import files
    files.download(f"notebooks/{NOTEBOOK}.ipynb")
    print(f"✓ Descarga iniciada: {NOTEBOOK}.ipynb")

In [ ]:
# Verificación rápida
from src.utils.io import load_embeddings
X_train, y_train = load_embeddings('train', embeddings_dir)
X_test, y_test   = load_embeddings('test',  embeddings_dir)

print(f'Train X: {X_train.shape} | dtype: {X_train.dtype}')
print(f'Test  X: {X_test.shape}  | dtype: {X_test.dtype}')
print(f'Rango de valores — min: {X_train.min():.3f} | max: {X_train.max():.3f}')